In [ ]:
import os

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as exc:
    print("Google Drive mount skipped/failed:", exc)

# Edit these two paths if your Drive layout is different.
DRIVE_ROOT = "/content/drive/MyDrive"
CHECKPOINT_ROOT = "/content/drive/MyDrive/Anh Khôi ĐACN/Thực nghiệm/100-clients/NICE_IL"
DATA_DIR = "/content/drive/MyDrive/Anh Khôi ĐACN/Dataset/2023/federated_splits/100-clients"

pt_files = []
for root, _, files in os.walk(CHECKPOINT_ROOT):
    for f in files:
        if f.endswith(".pt"):
            pt_files.append(os.path.join(root, f))

print("PT files found:")
for p in sorted(pt_files):
    print(p)

print("CHECKPOINT_ROOT:", CHECKPOINT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("DATA_DIR exists:", os.path.exists(DATA_DIR))


In [ ]:
import os

print("CHECKPOINT_ROOT:", CHECKPOINT_ROOT)
print("exists:", os.path.exists(CHECKPOINT_ROOT))

if os.path.exists(CHECKPOINT_ROOT):
    print("\nTop-level contents:")
    for name in sorted(os.listdir(CHECKPOINT_ROOT)):
        path = os.path.join(CHECKPOINT_ROOT, name)
        kind = "DIR " if os.path.isdir(path) else "FILE"
        size = os.path.getsize(path) if os.path.isfile(path) else ""
        print(f"{kind} {name} {size}")

    print("\n.pt/.pth preview under CHECKPOINT_ROOT:")
    count = 0
    for root, _, files in os.walk(CHECKPOINT_ROOT):
        for filename in sorted(files):
            if filename.lower().endswith((".pt", ".pth")):
                print(os.path.join(root, filename))
                count += 1
                if count >= 80:
                    print("... truncated at 80 files")
                    break
        if count >= 80:
            break
    print("pt/pth shown:", count)


In [ ]:
import json
import os
import re
import shutil
import sys
from collections import OrderedDict

REPO_PATH = "/tmp/FL_IL_IDS"
CHECKPOINT_ROOT = globals().get("CHECKPOINT_ROOT", "/content/drive/MyDrive/Anh Khôi ĐACN/Thực nghiệm/100-clients/NICE_IL")
DATA_DIR = globals().get("DATA_DIR", "/content/drive/MyDrive/100-clients/100-clients")
OUTPUT_DIR = "/content/eval_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def setup_imports():
    if os.path.exists(REPO_PATH):
        print(f"Removing stale clone at {REPO_PATH}...")
        shutil.rmtree(REPO_PATH)
    print("Cloning from GitHub...")
    os.system(f"git clone https://github.com/khoilv2005/FL_IL_IDS.git {REPO_PATH}")
    kaggle_prefix = "/kaggle/input"
    sys.path = [REPO_PATH] + [p for p in sys.path if not p.startswith(kaggle_prefix)]
    for name in list(sys.modules.keys()):
        if name == "fed_learning" or name.startswith("fed_learning."):
            del sys.modules[name]
    print("sys.path[0]:", sys.path[0])

setup_imports()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

from eval_checkpoint import _make_model
from fed_learning.data.incremental_loader import IncrementalDataLoader
from fed_learning.training.local_task_loop import (
    _apply_local_nice_context_mask,
    _nice_seen_mask,
)

def _print_tree_preview(root, max_dirs=30, max_files=80):
    print(f"CHECKPOINT_ROOT: {root}")
    print("exists:", os.path.exists(root))
    if not os.path.exists(root):
        return
    printed_dirs = 0
    printed_files = 0
    for dirpath, dirnames, filenames in os.walk(root):
        depth = dirpath[len(root):].count(os.sep)
        if depth > 2:
            dirnames[:] = []
            continue
        if printed_dirs < max_dirs:
            print("DIR ", dirpath)
            printed_dirs += 1
        for filename in filenames:
            if printed_files >= max_files:
                break
            print("FILE", os.path.join(dirpath, filename))
            printed_files += 1
        if printed_files >= max_files and printed_dirs >= max_dirs:
            break


def _checkpoint_score(path):
    name = os.path.basename(path)
    round_match = re.match(r"checkpoint_task_(\d+)_round_(\d+)\.pt$", name)
    if round_match:
        return int(round_match.group(1)), int(round_match.group(2)), os.path.getmtime(path), path
    task_match = re.match(r"checkpoint_task_(\d+)\.pt$", name)
    if task_match:
        return int(task_match.group(1)), -1, os.path.getmtime(path), path

    # Fallback for renamed checkpoints: inspect metadata without requiring filename.
    try:
        obj = torch.load(path, map_location="cpu", weights_only=False)
        if not isinstance(obj, dict) or "model_state_dict" not in obj:
            return None
        task = int(obj.get("task_id", obj.get("config", {}).get("task_end", -1)) or -1)
        round_id = int(obj.get("round_id", obj.get("final_round_id", -1)) or -1)
        return task, round_id, os.path.getmtime(path), path
    except Exception:
        return None


def find_final_round_checkpoint(root):
    if not os.path.exists(root):
        raise FileNotFoundError(f"CHECKPOINT_ROOT does not exist: {root}")

    pt_files = []
    for dirpath, _, files in os.walk(root):
        for filename in files:
            lower = filename.lower()
            if lower.endswith((".pt", ".pth")):
                pt_files.append(os.path.join(dirpath, filename))

    print(f"Found {len(pt_files)} .pt/.pth files under CHECKPOINT_ROOT")
    if pt_files:
        print("First checkpoint candidates:")
        for p in sorted(pt_files)[:30]:
            print("  ", p)

    scored = []
    for path in pt_files:
        score = _checkpoint_score(path)
        if score is not None:
            scored.append(score)

    if scored:
        return max(scored, key=lambda x: (x[0], x[1], x[2]))[3]

    _print_tree_preview(root)
    raise FileNotFoundError(
        f"No loadable checkpoint with model_state_dict found under {root}. "
        "Set CHECKPOINT_ROOT to the folder that contains checkpoint_task_*.pt files, "
        "or extract/upload the checkpoint files into that folder."
    )

checkpoint_path = find_final_round_checkpoint(CHECKPOINT_ROOT)
print("Selected checkpoint:", checkpoint_path)

ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
config = dict(ckpt["config"])
config["data_dir"] = DATA_DIR
ckpt["config"] = config

device = "cuda" if torch.cuda.is_available() else "cpu"
eval_batch_size = int(config.get("eval_batch_size", 32768))
num_classes = int(config.get("total_classes", config.get("num_classes", 34)))
task_id = int(ckpt.get("task_id", config.get("task_end", 0)))
round_id = ckpt.get("round_id", ckpt.get("final_round_id"))
algorithm = str(ckpt.get("algorithm", config.get("algorithm", ""))).lower()

data_loader = IncrementalDataLoader(data_dir=DATA_DIR)
final_task_id = data_loader.get_num_tasks() - 1
if task_id != final_task_id:
    print(f"WARNING: selected checkpoint task_id={task_id}, dataset final_task_id={final_task_id}")

test_X, test_y = data_loader.get_test_data(task_id, cumulative=True)
model, context_detector = _make_model(ckpt, device)
seen_classes = ckpt.get("seen_classes") or list(range(num_classes))
criterion = nn.CrossEntropyLoss(reduction="sum")

all_preds = []
all_targets = []
total_loss = 0.0
model.eval()
with torch.no_grad():
    for start in range(0, len(test_y), eval_batch_size):
        X_batch = test_X[start:start + eval_batch_size].to(device, non_blocking=True)
        y_batch = test_y[start:start + eval_batch_size].to(device, non_blocking=True)
        if (
            context_detector is not None
            and seen_classes is not None
            and hasattr(model, "get_output_and_context_activations")
        ):
            logits, context_activations = model.get_output_and_context_activations(X_batch)
        else:
            logits = model(X_batch)
            context_activations = None

        loss_logits = logits.clone()
        if context_detector is not None and seen_classes is not None:
            global_unseen = _nice_seen_mask(model, seen_classes, device)
            if len(global_unseen) == loss_logits.shape[1]:
                loss_logits[:, global_unseen] = float("-inf")
            pred_logits = _apply_local_nice_context_mask(
                model,
                logits,
                X_batch,
                context_detector,
                seen_classes,
                device,
                context_activations=context_activations,
            )
        else:
            pred_logits = logits

        total_loss += criterion(loss_logits, y_batch).item()
        all_preds.append(pred_logits.argmax(dim=1).detach().cpu().numpy())
        all_targets.append(y_batch.detach().cpu().numpy())
        del X_batch, y_batch, logits, pred_logits, loss_logits

y_true = np.concatenate(all_targets)
y_pred = np.concatenate(all_preds)
stored_metrics = ckpt.get("metrics", {}) or {}
metrics = OrderedDict(
    checkpoint=checkpoint_path,
    algorithm=algorithm,
    task_id=task_id,
    round_id=round_id,
    train_loss=stored_metrics.get("train_loss"),
    test_loss=total_loss / max(1, len(y_true)),
    accuracy=accuracy_score(y_true, y_pred),
    precision_macro=precision_score(y_true, y_pred, average="macro", labels=list(range(num_classes)), zero_division=0),
    recall_macro=recall_score(y_true, y_pred, average="macro", labels=list(range(num_classes)), zero_division=0),
    f1_macro=f1_score(y_true, y_pred, average="macro", labels=list(range(num_classes)), zero_division=0),
    f1_weighted=f1_score(y_true, y_pred, average="weighted", labels=list(range(num_classes)), zero_division=0),
)

metrics_path = os.path.join(OUTPUT_DIR, "final_round_metrics.json")
metrics_csv_path = os.path.join(OUTPUT_DIR, "final_round_metrics.csv")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2, default=str)
pd.DataFrame([metrics]).to_csv(metrics_csv_path, index=False)

labels = list(range(num_classes))
cm = confusion_matrix(y_true, y_pred, labels=labels)
cm_csv_path = os.path.join(OUTPUT_DIR, "confusion_matrix_34.csv")
cm_png_path = os.path.join(OUTPUT_DIR, "confusion_matrix_34.png")
cm_pdf_path = os.path.join(OUTPUT_DIR, "confusion_matrix_34.pdf")
pd.DataFrame(cm, index=labels, columns=labels).to_csv(cm_csv_path)

plt.figure(figsize=(24, 20))
sns.heatmap(cm, cmap="Blues", xticklabels=labels, yticklabels=labels, cbar=True)
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.title(f"Confusion Matrix - {num_classes} classes - task {task_id} round {round_id}")
plt.tight_layout()
plt.savefig(cm_png_path, dpi=200)
plt.savefig(cm_pdf_path)
plt.show()

print(json.dumps(metrics, indent=2, default=str))
print("Saved metrics JSON:", metrics_path)
print("Saved metrics CSV:", metrics_csv_path)
print("Saved confusion CSV:", cm_csv_path)
print("Saved confusion PNG:", cm_png_path)
print("Saved confusion PDF:", cm_pdf_path)


In [ ]:
import json
import os

import pandas as pd

# Scan all result folders and merge every final_round_metrics.json into one table.
METRICS_ROOTS = [
    globals().get("CHECKPOINT_ROOT", "/content/drive/MyDrive"),
    globals().get("OUTPUT_DIR", "/content/eval_outputs"),
]
MERGED_OUTPUT_DIR = globals().get("OUTPUT_DIR", "/content/eval_outputs")
os.makedirs(MERGED_OUTPUT_DIR, exist_ok=True)

rows = []
seen_paths = set()
for root_dir in METRICS_ROOTS:
    if not root_dir or not os.path.exists(root_dir):
        continue
    for root, _, files in os.walk(root_dir):
        if "final_round_metrics.json" not in files:
            continue
        path = os.path.join(root, "final_round_metrics.json")
        if path in seen_paths:
            continue
        seen_paths.add(path)
        try:
            with open(path, "r") as f:
                record = json.load(f)
            record["metrics_file"] = path
            record["run_dir"] = root
            rows.append(record)
        except Exception as exc:
            print("Failed to load:", path, exc)

rows = sorted(
    rows,
    key=lambda r: (
        str(r.get("algorithm", "")),
        int(r.get("task_id", -1) or -1),
        int(r.get("round_id", -1) or -1),
        str(r.get("checkpoint", "")),
    ),
)

merged_json = os.path.join(MERGED_OUTPUT_DIR, "all_final_round_metrics.json")
merged_csv = os.path.join(MERGED_OUTPUT_DIR, "all_final_round_metrics.csv")
with open(merged_json, "w") as f:
    json.dump(rows, f, indent=2, default=str)
pd.DataFrame(rows).to_csv(merged_csv, index=False)

print(f"Merged {len(rows)} final_round_metrics.json files")
print("Saved merged JSON:", merged_json)
print("Saved merged CSV:", merged_csv)
pd.DataFrame(rows).head()
